# 🌿 EcoClean AI — Classify Waste Products Using Transfer Learning
> **IBM AI Engineering Professional Certificate | Module 7 Final Project**  
> VGG16 Transfer Learning (Extract Features) + Fine-Tuning on Organic vs Recyclable Waste dataset

## Setup & Imports

In [ ]:
import os, warnings, zipfile, glob, requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import vgg16
from sklearn import metrics as skm

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


## Task 1 — Print the Version of TensorFlow

In [ ]:
print(tf.__version__)

## Hyperparameters & Paths

In [ ]:
# Paths
BASE_DIR   = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, 'o-vs-r-split')
TRAIN_PATH = os.path.join(DATA_DIR, 'train')
TEST_PATH  = os.path.join(DATA_DIR, 'test')
MODEL1     = os.path.join(BASE_DIR, 'O_R_tlearn_vgg16.keras')
MODEL2     = os.path.join(BASE_DIR, 'O_R_tlearn_fine_tune_vgg16.keras')

# Hyperparameters
img_rows, img_cols = 150, 150
batch_size = 32
seed       = 42
val_split  = 0.2
print('Paths & hyperparameters configured.')


## Download Dataset

In [ ]:
if not os.path.exists(DATA_DIR):
    url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/kd6057VPpABQ2FqCbgu9YQ/o-vs-r-split-reduced-1200.zip'
    zp  = os.path.join(BASE_DIR, 'data.zip')
    print('Downloading dataset...')
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(zp, 'wb') as f:
            for chunk in r.iter_content(8192): f.write(chunk)
    with zipfile.ZipFile(zp) as z: z.extractall(BASE_DIR)
    os.remove(zp)
    print('Dataset downloaded and extracted!')
else:
    print('Dataset already present.')


## Data Generators

In [ ]:
# Training generator (with augmentation)
train_datagen = ImageDataGenerator(
    validation_split=val_split, rescale=1/255.,
    width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)

# Validation generator
val_datagen = ImageDataGenerator(validation_split=val_split, rescale=1/255.)

# Test generator
test_datagen = ImageDataGenerator(rescale=1/255.)

train_generator = train_datagen.flow_from_directory(
    directory=TRAIN_PATH, class_mode='binary', seed=seed,
    batch_size=batch_size, shuffle=True,
    target_size=(img_rows, img_cols), subset='training')

validation_generator = val_datagen.flow_from_directory(
    directory=TRAIN_PATH, class_mode='binary', seed=seed,
    batch_size=batch_size, shuffle=True,
    target_size=(img_rows, img_cols), subset='validation')


## Task 2 — Create `test_generator` Using the `test_datagen` Object

In [ ]:
test_generator = test_datagen.flow_from_directory(
    directory=TEST_PATH, class_mode='binary', seed=seed,
    batch_size=batch_size, shuffle=False,
    target_size=(img_rows, img_cols))

print('Class indices:', test_generator.class_indices)
print('Total test images:', test_generator.n)


## Task 3 — Print the Length of `train_generator`

In [ ]:
print(len(train_generator))

## Build VGG16 Model (Extract Features)

In [ ]:
def build_model(fine_tune=False):
    vgg_base = vgg16.VGG16(include_top=False, weights='imagenet',
                            input_shape=(img_rows, img_cols, 3))
    flat = Flatten()(vgg_base.layers[-1].output)
    bm   = Model(vgg_base.input, flat)
    for layer in bm.layers: layer.trainable = False
    if fine_tune:
        unfrozen = False
        for layer in bm.layers:
            if layer.name == 'block5_conv3': unfrozen = True
            layer.trainable = unfrozen
    model = Sequential([
        bm,
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid'),
    ])
    return model

model = build_model(fine_tune=False)


## Task 4 — Print the Summary of the Model

In [ ]:
model.summary()

## Task 5 — Compile the Model

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    metrics=['accuracy'])

print('Model compiled successfully!')


## Learning Rate Schedule

In [ ]:
def exp_decay_lr_schedule(epoch):
    return 1e-4 * np.exp(-0.1 * epoch)


## Phase 1 — Train Extract Features Model

In [ ]:
history_extract = model.fit(
    train_generator,
    steps_per_epoch=5,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    callbacks=[
        LearningRateScheduler(exp_decay_lr_schedule),
        EarlyStopping(monitor='val_loss', patience=4, min_delta=0.01),
        ModelCheckpoint(MODEL1, monitor='val_loss', save_best_only=True),
    ],
    verbose=1)

extract_feat_model = tf.keras.models.load_model(MODEL1)
print('Phase 1 training complete — model saved.')


## Task 6 — Plot Accuracy Curves for Training and Validation Sets (extract_feat_model)

In [ ]:
acc     = history_extract.history['accuracy']
val_acc = history_extract.history['val_accuracy']
epochs  = range(1, len(acc) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, acc,     'b-o', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r-s', label='Validation Accuracy')
plt.title('Task 6 — Accuracy Curves: Extract Features Model')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Phase 2 — Train Fine-Tuned Model

In [ ]:
fine_tune_model = build_model(fine_tune=True)
fine_tune_model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=['accuracy'])

# Re-create generators so they start fresh
train_gen2 = ImageDataGenerator(
    validation_split=val_split, rescale=1/255.,
    width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True
).flow_from_directory(
    directory=TRAIN_PATH, class_mode='binary', seed=seed,
    batch_size=batch_size, shuffle=True,
    target_size=(img_rows, img_cols), subset='training')

val_gen2 = ImageDataGenerator(
    validation_split=val_split, rescale=1/255.
).flow_from_directory(
    directory=TRAIN_PATH, class_mode='binary', seed=seed,
    batch_size=batch_size, shuffle=True,
    target_size=(img_rows, img_cols), subset='validation')

history_fine = fine_tune_model.fit(
    train_gen2,
    steps_per_epoch=5,
    epochs=10,
    validation_data=val_gen2,
    validation_steps=val_gen2.samples // batch_size,
    callbacks=[
        LearningRateScheduler(exp_decay_lr_schedule),
        EarlyStopping(monitor='val_loss', patience=4, min_delta=0.01),
        ModelCheckpoint(MODEL2, monitor='val_loss', save_best_only=True),
    ],
    verbose=1)

fine_tune_model = tf.keras.models.load_model(MODEL2)
print('Phase 2 training complete — model saved.')


## Task 7 — Plot Loss Curves for Training and Validation Sets (fine tune model)

In [ ]:
loss     = history_fine.history['loss']
val_loss = history_fine.history['val_loss']
epochs_ft = range(1, len(loss) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_ft, loss,     'b-o', label='Training Loss')
plt.plot(epochs_ft, val_loss, 'r-s', label='Validation Loss')
plt.title('Task 7 — Loss Curves: Fine-Tuned Model')
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Task 8 — Plot Accuracy Curves for Training and Validation Sets (fine tune model)

In [ ]:
ft_acc     = history_fine.history['accuracy']
ft_val_acc = history_fine.history['val_accuracy']

plt.figure(figsize=(10, 5))
plt.plot(epochs_ft, ft_acc,     'b-o', label='Training Accuracy')
plt.plot(epochs_ft, ft_val_acc, 'r-s', label='Validation Accuracy')
plt.title('Task 8 — Accuracy Curves: Fine-Tuned Model')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Load Test Images for Prediction

In [ ]:
test_files_O = sorted(glob.glob(os.path.join(TEST_PATH, 'O', '*')))
test_files_R = sorted(glob.glob(os.path.join(TEST_PATH, 'R', '*')))
test_files   = test_files_O[:50] + test_files_R[:50]

test_imgs   = np.array([
    tf.keras.preprocessing.image.img_to_array(
        tf.keras.preprocessing.image.load_img(f, target_size=(img_rows, img_cols)))
    for f in test_files], dtype='float32') / 255.

test_labels = [Path(f).parent.name for f in test_files]
print(f'Loaded {len(test_imgs)} test images.')


## Task 9 — Plot a Test Image Using Extract Features Model (index_to_plot = 1)

In [ ]:
index_to_plot = 1

pred_ef  = extract_feat_model.predict(test_imgs, verbose=0)
label_ef = 'Recyclable (R)' if pred_ef[index_to_plot][0] >= 0.5 else 'Organic (O)'
actual   = test_labels[index_to_plot]

plt.figure(figsize=(5, 5))
plt.imshow((test_imgs[index_to_plot] * 255).astype('uint8'))
plt.title(f'Task 9 | Extract Features Model\nActual: {actual}  |  Predicted: {label_ef}', fontsize=12)
plt.axis('off')
plt.tight_layout(); plt.show()
print(f'Actual: {actual} | Predicted: {label_ef}')


## Task 10 — Plot a Test Image Using Fine-Tuned Model (index_to_plot = 1)

In [ ]:
pred_ft  = fine_tune_model.predict(test_imgs, verbose=0)
label_ft = 'Recyclable (R)' if pred_ft[index_to_plot][0] >= 0.5 else 'Organic (O)'

plt.figure(figsize=(5, 5))
plt.imshow((test_imgs[index_to_plot] * 255).astype('uint8'))
plt.title(f'Task 10 | Fine-Tuned Model\nActual: {actual}  |  Predicted: {label_ft}', fontsize=12)
plt.axis('off')
plt.tight_layout(); plt.show()
print(f'Actual: {actual} | Predicted: {label_ft}')
